# GRPO: Group Relative Policy Optimization (2024)

## The DeepSeek-R1 Moment

In January 2025, DeepSeek released R1 — a model that matched OpenAI o1 on reasoning benchmarks at a fraction of the training cost.
The key technique: **GRPO** (Group Relative Policy Optimization, Shao et al. 2024).

GRPO is a PPO variant that:
1. **Eliminates the critic network** — standard PPO needs a value function model (same size as policy). GRPO doesn't.
2. **Uses group-relative rewards** — instead of absolute rewards, normalizes within a group of completions for the same prompt
3. **Works with verifiable rewards** — math correctness, code execution, format checking — no human labeling needed

## GRPO Algorithm

For each prompt x, sample G completions {y₁, y₂, ..., y_G}:

```
Advantage of yᵢ = (reward(yᵢ) - mean(rewards)) / std(rewards)

GRPO loss = -E[ min(
    π(y|x)/π_old(y|x) · A(x,y),
    clip(π(y|x)/π_old(y|x), 1-ε, 1+ε) · A(x,y)
) ] + β · KL(π || π_ref)
```

Group normalization replaces the critic's value estimate — much more stable and memory-efficient.

In [ ]:
!pip install -U trl transformers datasets peft -q
import trl; print(f"TRL {trl.__version__}")

In [ ]:
import torch, gc, re
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig

gc.collect(); torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params | {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Designing Reward Functions

This is the engineering core of GRPO. Reward functions must:
- Accept `(prompts, completions, **kwargs)` where kwargs contains all dataset columns
- Return `List[float]` — one reward per completion
- Be fast (called every training step)

**Types of reward functions:**
1. **Accuracy rewards** — does the answer match ground truth?
2. **Format rewards** — is the output structured correctly?
3. **Code execution rewards** — does the code run without errors?
4. **Reasoning trace rewards** — does the model show its work?

In [ ]:
# ── Reward functions ─────────────────────────────────────────

def accuracy_reward(prompts, completions, answer, **kwargs):
    """
    Checks if the completion contains the correct numerical answer.
    Gives 1.0 for exact match, 0.0 otherwise.
    """
    rewards = []
    for completion, gt in zip(completions, answer):
        numbers = re.findall(r'\b\d+\b', completion)
        predicted = numbers[-1] if numbers else ""
        rewards.append(1.0 if predicted.strip() == gt.strip() else 0.0)
    return rewards

def format_reward(prompts, completions, **kwargs):
    """
    Rewards concise answers that contain a number.
    Teaches the model to be direct rather than verbose.
    """
    rewards = []
    for completion in completions:
        numbers = re.findall(r'\b\d+\b', completion)
        length = len(completion.strip())
        if numbers and length <= 10:      # short, contains number
            rewards.append(0.5)
        elif numbers and length <= 30:    # medium length
            rewards.append(0.3)
        elif numbers:                     # long but has number
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

# Verify reward functions behave correctly
test_completions = ["8", "The answer is 8, because 5+3=8", "eight", "I think it might be 8 or possibly 9"]
test_answers = ["8"] * 4
acc_r = accuracy_reward(["5+3=?"]*4, test_completions, test_answers)
fmt_r = format_reward(["5+3=?"]*4, test_completions)
print("Completion → Accuracy | Format")
for c, a, f in zip(test_completions, acc_r, fmt_r):
    print(f"  {c!r:45s} → {a:.1f} | {f:.1f}")

In [ ]:
# ── Dataset: arithmetic problems with ground truth answers ─────
math_data = [
    {"prompt": "What is 12 + 7? Answer with only the number.", "answer": "19"},
    {"prompt": "What is 25 - 8? Answer with only the number.",  "answer": "17"},
    {"prompt": "What is 6 × 4? Answer with only the number.",   "answer": "24"},
    {"prompt": "What is 100 / 5? Answer with only the number.", "answer": "20"},
    {"prompt": "What is 13 + 29? Answer with only the number.", "answer": "42"},
    {"prompt": "What is 50 - 23? Answer with only the number.", "answer": "27"},
    {"prompt": "What is 8 × 7? Answer with only the number.",   "answer": "56"},
    {"prompt": "What is 81 / 9? Answer with only the number.",  "answer": "9"},
] * 4  # 32 prompts

dataset = Dataset.from_list(math_data)
print(f"Training dataset: {len(dataset)} math problems")

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

grpo_config = GRPOConfig(
    output_dir="/tmp/grpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,          # G=4: sample 4 completions per prompt
    max_completion_length=32,   # short completions (just the number + brief explanation)
    fp16=True,
    logging_steps=4,
    save_strategy="no",
    report_to="none",
    use_vllm=False,             # standard generation; set True + vLLM installed for ~3x speed
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=[accuracy_reward, format_reward],  # both rewards combined
    train_dataset=dataset,
    peft_config=lora_config,
)

result = trainer.train()
print(f"\n✅ GRPO training complete!")
print(f"   Loss: {result.training_loss:.4f}")
print(f"   Runtime: {result.metrics['train_runtime']:.1f}s")
print(f"   Reward functions used: accuracy (0/1) + format (0-0.5)")

## Key Takeaways

1. **GRPO = PPO without critic** — group normalization replaces the value function, saving 50% memory
2. **Verifiable rewards are key** — math correctness, code execution, format rules = no human labeling
3. **num_generations (G)** — larger G = more stable gradient estimates but more compute
4. **Multiple reward functions** — combine accuracy + format + style rewards, each returned as List[float]
5. **DeepSeek-R1 used G=8, training on 300K math problems** — our toy example uses G=4, 32 problems

## Scale: What Full GRPO Training Looks Like

For a real reasoning model (following DeepSeek-R1-Zero pattern):
1. Load a 7B base model (no SFT, just pre-training)
2. Dataset: 300K math problems from OpenMathInstruct-2, GSM8K, MATH
3. Reward: format (`<think>...</think><answer>N</answer>`) + accuracy (extract and verify)
4. Training: 8× A100 80GB, ~$200 on Lambda Labs, ~12 hours
5. Result: emergent chain-of-thought reasoning without supervised CoT data